# Multimodal Models 3: Flamingo-Style Visual Question Answering

## Perceiver Resampler and gated cross-attention

Flamingo connects pretrained visual and language models so a language model can process interleaved text and images or video. A vision encoder produces a variable number of visual features. The **Perceiver Resampler** uses learned latent queries to compress them to a fixed number of visual tokens. **Gated cross-attention** layers inserted between language-model blocks let text features attend to those visual tokens. A learned gate, initialized near zero, allows visual conditioning to enter gradually while preserving the pretrained language behavior at initialization.

This is different from CLIP: CLIP aligns global image/text embeddings for comparison, while Flamingo injects visual features into a language model that generates text. The architecture also handles interleaved multimodal context; this small case study simplifies that setting to one image and one question. We use CIFAR-10 and ask “what object is shown?”; the answer is one of ten class names.

The notebook implements a compact Perceiver Resampler and gated cross-attention in PyTorch. All components are trained from scratch on a small dataset; Flamingo itself uses large pretrained backbones and large-scale multimodal training. CIFAR-10 or a procedural fallback keeps the example runnable.

## 1. Dataset and text query

Each image is paired with the same short question; its class label is the target answer. This task isolates whether the visual signal can flow through the resampler and cross-attention bridge. A synthetic fallback preserves the interface when torchvision or CIFAR-10 download is unavailable.

**What this cell does:** loads a small CIFAR-10 subset or colored-icon fallback, then creates image/question/answer batches.

In [ ]:
import random,numpy as np,torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,Subset
import matplotlib.pyplot as plt
SEED=22;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes=['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
try:
 from torchvision.datasets import CIFAR10
 from torchvision import transforms
 base=CIFAR10('./data',train=True,download=True,transform=transforms.ToTensor())
except Exception as e:
 print('Using synthetic fallback:',type(e).__name__)
 class Icons(Dataset):
  def __len__(self):return 1200
  def __getitem__(self,i):
   y=i%10;x=torch.rand(3,32,32)*.08;x[y%3,8:24,8:24]+=.8;return x.clamp(0,1),y
 base=Icons()
ix=np.random.permutation(len(base))[:min(1800,len(base))];ds=Subset(base,ix);loader=DataLoader(ds,batch_size=64,shuffle=True,drop_last=True)
print('examples:',len(ds),'device:',device,'question: what object is shown?')


## 2. Vision encoder and Perceiver Resampler

The compact vision encoder converts an image to a grid of features. The resampler maintains a learned set of latent queries and cross-attends them to the image features, producing a fixed number of visual tokens regardless of input feature count. A feed-forward block refines each latent.

**What this cell does:** defines the patch-feature encoder and two-layer latent resampler, then checks the output token shape.

In [ ]:
WIDTH=128;N_LATENTS=8
class VisionEncoder(nn.Module):
 def __init__(self):
  super().__init__();self.net=nn.Sequential(nn.Conv2d(3,64,4,4),nn.GELU(),nn.Conv2d(64,WIDTH,1))
 def forward(self,x):return self.net(x).flatten(2).transpose(1,2)
class Resampler(nn.Module):
 def __init__(self):
  super().__init__();self.latents=nn.Parameter(torch.randn(1,N_LATENTS,WIDTH)*.02);self.attn=nn.ModuleList([nn.MultiheadAttention(WIDTH,4,batch_first=True) for _ in range(2)]);self.norm=nn.ModuleList([nn.LayerNorm(WIDTH) for _ in range(2)]);self.ff=nn.ModuleList([nn.Sequential(nn.LayerNorm(WIDTH),nn.Linear(WIDTH,4*WIDTH),nn.GELU(),nn.Linear(4*WIDTH,WIDTH)) for _ in range(2)])
 def forward(self,x):
  z=self.latents.expand(x.size(0),-1,-1)
  for a,n,f in zip(self.attn,self.norm,self.ff):z=z+a(n(z),x,x,need_weights=False)[0];z=z+f(z)
  return z
vision=VisionEncoder().to(device);resampler=Resampler().to(device)
image,_=next(iter(loader));features=vision(image.to(device));tokens=resampler(features);print('patches:',features.shape,'resampled visual tokens:',tokens.shape)


## 3. Gated cross-attention and language-side answer head

Text tokens query the resampled visual tokens. The cross-attention output is multiplied by `tanh(gate)` before a residual addition. Initializing the scalar gate at zero starts with no visual perturbation; training can open the gate as needed. We use a tiny learned question-token sequence and an answer classifier instead of a full autoregressive language model.

**What this cell does:** implements gated cross-attention, a small text stream, and a classification head that predicts the object named in the image.

In [ ]:
class GatedCrossAttention(nn.Module):
 def __init__(self):
  super().__init__();self.norm=nn.LayerNorm(WIDTH);self.attn=nn.MultiheadAttention(WIDTH,4,batch_first=True);self.gate=nn.Parameter(torch.zeros(()))
 def forward(self,text,visual):
  q=self.norm(text);return text+torch.tanh(self.gate)*self.attn(q,visual,visual,need_weights=False)[0]
class FlamingoMini(nn.Module):
 def __init__(self):
  super().__init__();self.vision=VisionEncoder();self.resampler=Resampler();self.words=nn.Embedding(5,WIDTH);self.cross=GatedCrossAttention();self.head=nn.Sequential(nn.LayerNorm(WIDTH),nn.Linear(WIDTH,10))
  self.register_buffer('question',torch.tensor([1,2,3,4,0]))
 def forward(self,image):
  vis=self.resampler(self.vision(image));txt=self.words(self.question)[None].expand(len(image),-1,-1);fused=self.cross(txt,vis);return self.head(fused.mean(1))
model=FlamingoMini().to(device)
print('gated cross-attention init:',float(model.cross.gate),'parameters:',sum(p.numel() for p in model.parameters()))


## 4. Train the image-grounded answer model

We optimize cross-entropy over ten answer classes. Because the question is fixed, the model's ability to answer depends on visual information passing through the cross-attention path. The gate starts at zero and learns from the classification objective. This simple supervised task does not test few-shot language generation or multi-image reasoning.

**What this cell does:** trains the model for a few epochs and reports validation accuracy and the learned gate value.

In [ ]:
train_ds=Subset(ds,range(int(.85*len(ds))));val_ds=Subset(ds,range(int(.85*len(ds)),len(ds)))
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,drop_last=True);val_loader=DataLoader(val_ds,batch_size=64)
opt=torch.optim.AdamW(model.parameters(),lr=2e-3);EPOCHS=5
for ep in range(EPOCHS):
 model.train();ls=[]
 for x,y in train_loader:
  x,y=x.to(device),y.to(device);logits=model(x);loss=F.cross_entropy(logits,y);opt.zero_grad(set_to_none=True);loss.backward();opt.step();ls.append(loss.item())
 model.eval();correct=total=0
 with torch.no_grad():
  for x,y in val_loader:
   logits=model(x.to(device));correct+=(logits.argmax(1).cpu()==y).sum().item();total+=len(y)
 print(f'epoch {ep+1}: train loss={np.mean(ls):.3f}, validation accuracy={correct/total:.3f}, gate={model.cross.gate.item():.3f}')


## 5. Inspect answers and cross-modal attention

We inspect a few image/question/answer predictions. The attention map in this simplified model shows how each question token distributes attention over the fixed visual latent tokens. It is a diagnostic of the bridge, not a faithful explanation of the model's reasoning.

**What this cell does:** displays predictions and visual-token attention weights for one example.

In [ ]:
model.eval();x,y=next(iter(val_loader));x=x.to(device)
with torch.no_grad():
 vis=model.resampler(model.vision(x));txt=model.words(model.question)[None].expand(len(x),-1,-1);q=model.cross.norm(txt);_,weights=model.cross.attn(q,vis,vis,need_weights=True,average_attn_weights=True);pred=model(x).argmax(1).cpu()
fig,ax=plt.subplots(1,6,figsize=(12,2.5))
for i,a in enumerate(ax):a.imshow(x[i].cpu().permute(1,2,0));a.set_title(f'true {classes[int(y[i])]}/pred {classes[int(pred[i])]}' ,fontsize=7);a.axis('off')
plt.tight_layout();plt.show()
plt.imshow(weights[0].cpu(),aspect='auto',cmap='viridis');plt.xlabel('resampled visual token');plt.ylabel('question token');plt.title('Cross-attention weights');plt.colorbar();plt.show()


## Key takeaways

- Flamingo bridges vision and language with a fixed-size Perceiver Resampler and gated cross-attention inside a language model.
- The resampler compresses variable-length visual features into a fixed number of tokens.
- A near-zero gate allows visual information to enter gradually through residual cross-attention.
- This notebook demonstrates the interface on a small image-question task; it is not a pretrained Flamingo reproduction and does not model interleaved video or few-shot generation.
- Evaluate multimodal systems for grounding, hallucination, robustness, and data leakage—not only classification accuracy.

**Reference:** [Flamingo: a Visual Language Model for Few-Shot Learning](https://arxiv.org/abs/2204.14198).